<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

The checkpoint is the reparameterized inference graph. Each dilated large-kernel branch is one depthwise 9×9 plus a pointwise mix, and the IntraCL batch-norm is folded into the closing 1×1. Submodules are stored in the order the exporter wrote them. Changing that order will not fail at construction. It fails when a checkpoint tensor has the wrong shape.

In [4]:
#| echo: false
#| output: asis
show_doc(LargeKernel)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/layers/pan.py#L22){target="_blank" style="float:right; font-size:smaller"}

### LargeKernel

```python
def LargeKernel(
    in_channels:int, out_channels:int, kernel:int
)->None:
```

*Depthwise convolution followed by a pointwise mix.*

## IntraCL

In [6]:
#| echo: false
#| output: asis
show_doc(IntraCL)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/layers/pan.py#L42){target="_blank" style="float:right; font-size:smaller"}

### IntraCL

```python
def IntraCL(
    channels:int, reduce_factor:int=2
)->None:
```

*Strip and square context at 7, 5, and 3, added back onto the input.*

At each scale the square kernel and the two strips (vertical, then horizontal) are added, then the next smaller scale reads that sum. The projection is a residual on the block input. Attribute order is the checkpoint order, including the 7-then-5-then-3 groups.

## The neck

In [8]:
#| echo: false
#| output: asis
show_doc(RepLKPAN)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/layers/pan.py#L69){target="_blank" style="float:right; font-size:smaller"}

### RepLKPAN

```python
def RepLKPAN(
    in_channels:tuple[int, int, int, int], out_channels:int, kernel:int
)->None:
```

*Top-down fusion, a bottom-up path, then IntraCL. Four levels are concatenated at the finest resolution.*

Top-down first: the coarsest lateral is upsampled onto the next finer map. Then a bottom-up path downsamples the fine mix back onto the coarse ones. IntraCL runs on each level, and the four maps are concatenated at the finest resolution. The output width is `fpn_channels`, because each level was projected to `fpn_channels // 4`.